In [1]:
import os
import asyncio
import aiohttp
import requests

import pandas as pd
import xarray as xr
import pygrib as pg
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import astropy.units as u

from pathlib import Path
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from matplotlib.colors import BoundaryNorm, ListedColormap
from astropy.time import Time
from astropy.coordinates import AltAz, EarthLocation, get_sun

## Using `pygrib`

In [3]:
grib_file = sorted(os.listdir("latest_wrf_files"))[0]
print(f"Opening GRIB file: {grib_file}")
grbs = pg.open("latest_wrf_files/" + grib_file)

Opening GRIB file: WRF_cpt_07KM_2026010200_2026010201.grib2


In [4]:
grbs.messages

305

In [5]:
grb = grbs.message(1)
print(grb)

1:Pressure reduced to MSL:Pa (instant):regular_ll:meanSea:level 0:fcst time 1 hrs:from 202601020000


In [6]:
records = []

for g in grbs:
    records.append({
        "shortName": g.shortName,
        "name": g.name,
        "units": g.units,
        "typeOfLevel": g.typeOfLevel,
        "level": g.level,
        "forecastTime_h": g.forecastTime,
        "validDate": g.validDate,
        "gridType": g.gridType,
        "centre": g.centre
    })

grbs.close()

df = pd.DataFrame(records)

In [8]:
df

,shortName,name,units,typeOfLevel,level,forecastTime_h,validDate,gridType,centre
0,refc,Maximum/Composite radar reflectivity,dB,atmosphereSingleLayer,0,1,2026-01-02 01:00:00,regular_ll,kwbc
1,vis,Visibility,m,surface,0,1,2026-01-02 01:00:00,regular_ll,kwbc
2,vis,Visibility,m,cloudTop,0,1,2026-01-02 01:00:00,regular_ll,kwbc
3,gust,Wind speed (gust),m s**-1,surface,0,1,2026-01-02 01:00:00,regular_ll,kwbc
4,gh,Geopotential height,gpm,isobaricInhPa,50,1,2026-01-02 01:00:00,regular_ll,kwbc
...,...,...,...,...,...,...,...,...,...
299,cape,Convective available potential energy,J kg**-1,pressureFromGroundLayer,25500,1,2026-01-02 01:00:00,regular_ll,kwbc
300,cin,Convective inhibition,J kg**-1,pressureFromGroundLayer,25500,1,2026-01-02 01:00:00,regular_ll,kwbc
301,plpl,Pressure of level from which parcel was lifted,Pa,pressureFromGroundLayer,25500,1,2026-01-02 01:00:00,regular_ll,kwbc
302,lsm,Land-sea mask,(0 - 1),surface,0,1,2026-01-02 01:00:00,regular_ll,kwbc


In [10]:
df.to_csv("grib_messages_overview.csv", index=False)

In [9]:
for name in df["name"].unique():
    print(name)

Maximum/Composite radar reflectivity
Visibility
Wind speed (gust)
Geopotential height
Temperature
Potential temperature
Relative humidity
Dew point temperature
Specific humidity
Vertical velocity
U component of wind
V component of wind
MSLP (Eta model reduction)
Derived radar reflectivity
Surface pressure
Orography
Soil temperature
Volumetric soil moisture content
Liquid volumetric soil moisture (non-frozen)
Water equivalent of accumulated snow depth (deprecated)
Snow cover
2 metre temperature
2 metre specific humidity
2 metre dewpoint temperature
2 metre relative humidity
10 metre U wind component
10 metre V wind component
Total Precipitation
Convective precipitation (water)
Large scale precipitation (non-convective)
Storm surface runoff
Baseflow-groundwater runoff
Instantaneous surface sensible heat net flux
Surface latent heat net flux
Ground heat flux
Surface lifted index
Convective available potential energy
Convective inhibition
Precipitable water
Low cloud cover
Medium cloud cov

- Visibility
- Wind speed (gust)
- Temperature
- Relative humidity
- Dew point temperature
- Surface pressure
- Total Precipitation
- Low cloud cover
- Medium cloud cover
- High cloud cover
- Total Cloud Cover
- Pressure

In [ ]:
df[df["name"] == "Total Cloud Cover"]

In [ ]:
df["forecastTime_h"].unique()

In [ ]:
df[df["forecastTime_h"] == 0]

In [ ]:
grbs.select(name="Total Cloud Cover")

In [ ]:
# grbs = pg.open("WRF_cpt_07KM_2025110800_2025110803.grib2")

# Select the message containing "Total Cloud Cover"
# You can also use grbs.select(shortName='tcc') if you know the short name
grb = grbs.select(name='Total Cloud Cover')[0]

# Get the data grid and its lat/lon arrays
data, lats, lons = grb.data()

# Target coordinate
target_lat, target_lon = -25.39, -49.23

# Find the nearest grid point to that coordinate
distance = (lats - target_lat)**2 + (lons - target_lon)**2
iy, ix = np.unravel_index(distance.argmin(), distance.shape)

# Extract the value at that grid point
value = data[iy, ix]

print(f"Total Cloud Cover at ({target_lat}, {target_lon}) = {value:.2f} {grb.units}")

## Next 24 hours Cloud Cover with `pygrib`

In [ ]:
grbs = pg.open("WRF_cpt_07KM_2025110800_2025110803.grib2")

# Select the message containing "Total Cloud Cover"
# You can also use grbs.select(shortName='tcc') if you know the short name
grb = grbs.select(name='Total Cloud Cover')[0]

# Get the data grid and its lat/lon arrays
data, lats, lons = grb.data()

# Target coordinate
target_lat, target_lon = -25.39, 310.77

# Find the nearest grid point to that coordinate
distance = (lats - target_lat)**2 + (lons - target_lon)**2
iy, ix = np.unravel_index(distance.argmin(), distance.shape)

# Extract the value at that grid point
value = data[iy, ix]

print(f"Total Cloud Cover at ({target_lat}, {target_lon}) = {value:.2f} {grb.units}")

In [ ]:
grbs = pg.open("WRF_cpt_07KM_2025110800_2025110803.grib2")
grb = grbs.select(name='Total Cloud Cover')[0]

In [ ]:
grbs = pg.open("WRF_cpt_07KM_2025110800_2025110803.grib2")

# Select the message containing "Total Cloud Cover"
# You can also use grbs.select(shortName='tcc') if you know the short name
grb = grbs.select(name='Total Cloud Cover')[0]

# Get the data grid and its lat/lon arrays
data, lats, lons = grb.data()

In [ ]:
data.shape

In [ ]:
lats.min(), lats.max(), lons.min(), lons.max()

In [ ]:
import wget

# Base info
year = 2025
month = 11
day = 11
run_hour = "00"

# Forecast range (01 to 23)
forecast_hours = range(1, 24)


tcc_list = []
hour_list = []

# Loop to download each file
for fh in forecast_hours:
    # Format forecast hour as two digits
    fh_str = f"{fh:02d}"

    # Construct full URL
    url = (f"https://dataserver.cptec.inpe.br/dataserver_modelos/wrf/ams_07km/brutos/"
           f"{year}/{month:02d}/{day:02d}/{run_hour}/"
           f"WRF_cpt_07KM_{year}{month:02d}{day:02d}{run_hour}_"
           f"{year}{month:02d}{day:02d}{fh_str}.grib2")

    print("Downloading:", url)
    wget.download(url)

    grib_file = (f"WRF_cpt_07KM_{year}{month:02d}{day:02d}{run_hour}_"
                 f"{year}{month:02d}{day:02d}{fh_str}.grib2")
    grbs = pg.open(grib_file)

    # Select the message containing "Total Cloud Cover"
    # You can also use grbs.select(shortName='tcc') if you know the short name
    grb = grbs.select(name='Total Cloud Cover')[0]
    
    # Get the data grid and its lat/lon arrays
    data, lats, lons = grb.data()
    
    # Target coordinate
    target_lat, target_lon = -25.39, 310.77
    
    # Find the nearest grid point to that coordinate
    distance = (lats - target_lat)**2 + (lons - target_lon)**2
    iy, ix = np.unravel_index(distance.argmin(), distance.shape)
    
    # Extract the value at that grid point
    value = data[iy, ix]
    tcc_list.append(value)
    hour_list.append(fh)

In [ ]:
print(list(zip(tcc_list, hour_list)))

In [ ]:
plt.plot(hour_list, tcc_list, marker='o')
plt.xlabel("Forecast Hour")
plt.ylabel("Total Cloud Cover (%)")

## **Main application**

In [ ]:
# functions

# Configuration parameters
MAX_CONCURRENT = 12      # number of parallel downloads
CHUNK_SIZE = 1024 * 64  # 64 KB chunks
RETRIES = 3              # retries per file
TIMEOUT = aiohttp.ClientTimeout(
    total=None,            # no limit on total download time
    sock_connect=30,       # how long to wait to connect
    sock_read=60           # how long to wait for each chunk
)


# Download a single file
async def download_file(session, url, out_path):
    for attempt in range(1, RETRIES + 1):
        try:
            async with session.get(url, timeout=TIMEOUT) as resp:

                # File does not exist on server
                if resp.status == 404:
                    print(f"❌ Not found (404): {url}")
                    return

                # Other HTTP errors
                if resp.status != 200:
                    print(f"❌ HTTP {resp.status} for {url} (attempt {attempt})")
                    continue

                # Save file
                out_path.parent.mkdir(parents=True, exist_ok=True)
                with open(out_path, "wb") as f:
                    print(f"⬇️ Downloading: {out_path.name}")
                    async for chunk in resp.content.iter_chunked(CHUNK_SIZE):
                        f.write(chunk)

                print(f"✅ Downloaded: {out_path.name}")
                return
            
        except asyncio.TimeoutError:
            print(f"⚠️ Timeout downloading {url} (attempt {attempt})")
        except Exception as e:
            print(f"⚠️ Error downloading {url} (attempt {attempt}): {e}")
            print(f"Removing incomplete file {out_path.name}")
            os.remove(out_path) if out_path.exists() else None

    print(f"❌ Failed after {RETRIES} attempts: {url}")


# Download many files async
async def download_many(urls, output_dir):

    output_dir = Path(output_dir)
    connector = aiohttp.TCPConnector(limit=MAX_CONCURRENT)

    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = []
        for url in urls:
            filename = url.split("/")[-1]
            out_path = output_dir / filename

            # Skip if file already exists
            if out_path.exists():
                print(f"⏩ Skipped (exists): {filename}")
                continue

            tasks.append(download_file(session, url, out_path))

        if not tasks:
            print("All files already exist.")
            return

        await asyncio.gather(*tasks)


# get latest available forecast run hour (in UTC) for a given date
def get_latest_forecast_run_hour(date: datetime):
    latest = None
    for run_hour in range(24):
        url = (f"https://dataserver.cptec.inpe.br/dataserver_modelos/wrf/ams_07km/brutos/"
               f"{date.year}/{date.month:02d}/{date.day:02d}/{run_hour:02d}/")
        r = requests.head(url)
        if r.status_code in [200, 403]:
            latest = run_hour
    if latest is not None:
        return latest
    raise ValueError(f"No available forecast found for date {date.year}-{date.month:02d}-{date.day:02d}")


async def update_forecast_data():
    utc_now = datetime.now(timezone.utc)
    try:
        latest_run_hour = get_latest_forecast_run_hour(utc_now)
        initial_date_time = datetime(utc_now.year, utc_now.month, utc_now.day, latest_run_hour)
    except ValueError as e:
        print(e)
        yesterday = utc_now - timedelta(days=1)
        print(f"Trying yesterday's date ({yesterday.year}-{yesterday.month:02d}-{yesterday.day:02d})")
        latest_run_hour = get_latest_forecast_run_hour(yesterday)
        initial_date_time = datetime(yesterday.year, yesterday.month, yesterday.day, latest_run_hour)

    files_to_download = []
    for hour_delta in range(1, (24*7 + 1)):
        forecast_date_time = initial_date_time + timedelta(hours=hour_delta)

        url_path = (
            f"https://dataserver.cptec.inpe.br/dataserver_modelos/wrf/ams_07km/brutos/"
            f"{initial_date_time.year}/{initial_date_time.month:02d}/{initial_date_time.day:02d}/{initial_date_time.hour:02d}/"
        )        
        url_file_name = (
            f"WRF_cpt_07KM_{initial_date_time.year}{initial_date_time.month:02d}{initial_date_time.day:02d}{initial_date_time.hour:02d}_"
            f"{forecast_date_time.year}{forecast_date_time.month:02d}{forecast_date_time.day:02d}{forecast_date_time.hour:02d}.grib2"
        )
        url = url_path + url_file_name

        files_to_download.append(url)

    if os.path.exists("./data_tmp"):
        current_downloaded_files = os.listdir("./data_tmp")
    else:
        current_downloaded_files = []

    await download_many(files_to_download, "./data_tmp")

    # delete all 'old forecast files' in ./data_tmp
    for file in current_downloaded_files:
        full_path = os.path.join("./data_tmp", file)
        if full_path not in ["./data_tmp/" + url.split("/")[-1] for url in files_to_download]:
            print("Removing old file:", full_path)
            os.remove(full_path)


def get_tcc(target_lat: float, target_lon: float):
    tcc_values = []
    hour_list = []

    forecast_files = sorted([os.path.join("./latest_wrf_files", f) for f in os.listdir("./latest_wrf_files")])
    for file in forecast_files:
        try:
            grbs = pg.open(file)

            # Select the message containing "Total Cloud Cover"
            grb = grbs.select(name='Total Cloud Cover')[0]

            # Find the nearest grid point to target coordinate
            data, lats, lons = grb.data()
            distance = (lats - target_lat)**2 + (lons - target_lon)**2
            iy, ix = np.unravel_index(distance.argmin(), distance.shape)

            # Extract the value at that grid point
            tcc_values.append(data[iy, ix])
            hour_list.append(grb.validDate.replace(tzinfo=timezone.utc))
        except Exception as e:
            print(f"Error processing file {file}: {e}")
            continue
    
    return tcc_values, hour_list


def get_sunrise_sunset(date: datetime, lat, lon, elevation=0, timezone=timezone.utc):
    """
    Compute sunrise and sunset for given date and location.
    """
    location = EarthLocation.from_geodetic(lat=lat, lon=lon, height=elevation)

    # Sample times every 2 minutes across the whole day
    date = date.strftime("%Y-%m-%d")
    times = Time(f"{date} 00:00:00") + np.linspace(0, 24, 24*300)*u.hour

    # Sun altitude for each sample
    altaz = get_sun(times).transform_to(AltAz(obstime=times, location=location, pressure=1000*u.hPa))
    altitudes = altaz.alt.deg
    altitudes_corrected = altitudes + 0.833  # atmospheric refraction + solar radius for sunrise

    # Find index where altitude crosses 0 degrees (sunrise/set)
    sign = np.sign(altitudes_corrected)
    crossings = np.where(np.diff(sign))[0]

    if len(crossings) < 2:
        return None, None  # Polar day or night

    sunrise_time = times[crossings[0]].to_datetime(timezone=timezone)
    sunset_time  = times[crossings[1]].to_datetime(timezone=timezone)

    return sunrise_time, sunset_time

In [ ]:
# await update_forecast_data()

In [ ]:
# from the local grib2 files get cloud cover forecast for target location
target_lat, target_lon = -25.39, 310.79
tcc, forcast_time_utc = get_tcc(target_lat, target_lon)

In [ ]:
# convert datetimes in utc to Sao Paulo time
time_sp = np.array(forcast_time_utc)
time_sp = np.vectorize(lambda t: t.astimezone(ZoneInfo("America/Sao_Paulo")))(time_sp)

In [ ]:
data = list(zip(time_sp, tcc))

# only keep data from 5 hours before now to end of forecast
dawn_time = datetime.now(tz=ZoneInfo("America/Sao_Paulo")) - timedelta(hours=5)
data = [(t, v) for t, v in data if t >= dawn_time]
time_i, tcc_i = zip(*data)

In [ ]:
now = datetime.now(tz=ZoneInfo("America/Sao_Paulo"))
max_date = np.max(time_i)
(max_date - now).days

In [ ]:
fig, ax = plt.subplots(figsize=(10*7, 5))

ax.plot(time_i, tcc_i)

now = datetime.now(tz=ZoneInfo("America/Sao_Paulo"))
ax.axvline(now, lw=1, ls="-", c="red")
max_date = np.max(time_i)
max_day_delta = (max_date - now).days + 1
for day_delta in range(1, max_day_delta):
    d = now.replace(hour=0, minute=0, second=0) + timedelta(days=day_delta)
    ax.axvline(d, ls="-", c="k", alpha=1.0)
    sunset, sunrise = get_sunrise_sunset(d, target_lat, target_lon, timezone=ZoneInfo("America/Sao_Paulo"))
    ax.axvline(sunset, ls=":", c="orange", alpha=0.7)
    ax.axvline(sunrise, ls=":", c="orange", alpha=0.7)

ax.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=ZoneInfo("America/Sao_Paulo")))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H', tz=ZoneInfo("America/Sao_Paulo")))


In [ ]:
fig, ax = plt.subplots(figsize=(10*7, 2))

# edit blue color map
base_cmap = plt.get_cmap("Blues_r")
colors = base_cmap(np.linspace(0, 1.0, 10))  # pick 5 shades (avoid very light/white)
cmap = ListedColormap(colors)
bounds = range(0, 110, 10) # Define the bin edges (6 edges for 5 bins)
norm = BoundaryNorm(bounds, cmap.N)

# Convert y into a 2D array with shape (1, N) for imshow
y2d = np.array(tcc).reshape(1, -1)

# Show as colored squares
img = ax.imshow(
    y2d,
    aspect='auto',
    cmap=cmap,
    extent=[mdates.date2num(time_sp[0]), mdates.date2num(time_sp[-1] + timedelta(hours=1)), 0, 1]
)

ax.axvline(datetime.now(tz=ZoneInfo("America/Sao_Paulo")), lw=1, ls="-", c="red")
min_date = np.min(time_sp).replace(hour=0)
for day in range(1, 8):
    ax.axvline(min_date + timedelta(days=day), ls="-", c="k", alpha=1.0)
    sunset, sunrise = get_sunrise_sunset(min_date + timedelta(days=day-1), target_lat, target_lon, timezone=ZoneInfo("America/Sao_Paulo"))
    ax.axvline(sunset, ls=":", c="orange", alpha=1.0)
    ax.axvline(sunrise, ls=":", c="orange", alpha=1.0)

# X-axis formatting
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=ZoneInfo("America/Sao_Paulo")))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H', tz=ZoneInfo("America/Sao_Paulo")))

# Y-axis removed (it's just a single row)
ax.set_yticks([])

fig.colorbar(img, label="Total Cloud Cover (%)")
# plt.tight_layout()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta
import plotly.io as pio

# --- Create time axis ---
now = datetime.now()
future = now + timedelta(days=5)
times = pd.date_range(start=now, end=future, freq="1H")

# --- Sine wave with noise ---
y = np.sin(np.linspace(0, 10, len(times))) + np.random.normal(0, 0.2, len(times))

# --- Figure ---
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=times,
        y=y,
        mode="lines",
        name="Sine + Noise"
    )
)
fig.add_trace(
    go.Scatter(
        x=times,
        y=y,
        mode="lines",
        name="Sine + Noise"
    )
)

# --- Primary x-axis (bottom) ---
fig.update_xaxes(
    title="Time (bottom)",
    showgrid=True,
)

# --- Secondary x-axis (top) ---
fig.update_layout(
    xaxis2=dict(
        matches='x',
        overlaying='x',
        side='top',
        showline=True,          # ← REQUIRED so the axis shows up
        showticklabels=True,    # ← REQUIRED, otherwise invisible
        ticks="outside",
        linewidth=1,
        linecolor="black",
        tickformat="%a %d %b",  # Mon 02 Dec
        title="Day / Month (top)",
    )
)

fig.data[1].update(xaxis="x2")
fig.update_layout(
    title="Plot with Primary and Secondary X Axes",
    yaxis_title="Value",
)

fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig=make_subplots(
        specs=[[{"secondary_y": True}]])
print(fig.layout)    

fig.update_layout(xaxis2= {'anchor': 'y', 'overlaying': 'x', 'side': 'top'},
                  yaxis_domain=[0, 0.94]);

fig.add_trace(
    go.Bar(x=[1, 2, 3, 4],
           y=[7, 4, 5, 6],
           name="bar",
          ), secondary_y=False)
fig.add_trace(
    go.Scatter(x=[-2, -1, 0, 1],
               y=[4, 2, 5, 6],
               name="scatt1",
               line_color="#ee0000"), secondary_y=True)
fig.data[1].update(xaxis='x2')
fig.update_layout(width=700, height=475)